In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

model = VitsModel.from_pretrained("facebook/mms-tts-rhg")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-rhg")

text = ":Manúic beggún azad hísafe, ar izzot arde hók ókkol ót, fúainna hísafe foida óiye. Fottí insán óttu honó forók sára elan ot aséde tamám hók ókkol arde azadi ókkol loi fáaida goróon ór hók asé. Ar, taráre dil arde demak diyé. Ótolla, taráttu ekzon loi arekzon bái hísafe maamela goróon saá. "
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform

print('done')


In [ ]:
from openai import OpenAI

# OpenRouter client — uses the OpenAI-compatible endpoint
openrouter_client = OpenAI(
    api_key=OPENROUTER_API,
    base_url="https://openrouter.ai/api/v1",
)

def translate_to_rohingya(text: str) -> str:
    """Translate text to Rohingya Latin script using Gemini 2.5 via OpenRouter."""
    response = openrouter_client.chat.completions.create(
        model="google/gemini-3-pro-preview",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a translation assistant. "
                    "Translate the user's text into Rohingya written in Latin script (Rohingyalish). "
                    "Return only the translated text with no explanations or extra formatting."
                ),
            },
            {"role": "user", "content": text},
        ],
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()

english_text = "All human beings are born free and equal in dignity and rights."
rohingya_text = translate_to_rohingya(english_text)
print(f"English : {english_text}")
print(f"Rohingya: {rohingya_text}")

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
import numpy as np
import functools
import scipy.io.wavfile

ASR_MODEL_ID = "facebook/mms-1b-all"
ROHINGYA_LANG = "rhg"

@functools.lru_cache(maxsize=1)
def load_rohingya_asr_model():
    """Load MMS-1b-all ASR model with Rohingya adapter (cached)."""
    asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_ID)
    asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_ID)
    asr_processor.tokenizer.set_target_lang(ROHINGYA_LANG)
    asr_model.load_adapter(ROHINGYA_LANG)
    asr_model.eval()
    return asr_processor, asr_model


def transcribe_rohingya(audio: np.ndarray, sample_rate: int) -> str:
    """Convert Rohingya speech (numpy float32 array) to Rohingya Latin text."""
    asr_processor, asr_model = load_rohingya_asr_model()
    inputs = asr_processor(audio, sampling_rate=sample_rate, return_tensors="pt")
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    return asr_processor.decode(predicted_ids[0])


def translate_audio_to_english(wav_path: str) -> str:
    """
    Takes a path to a Rohingya .wav file and returns the English translation.

    Steps:
      1. Read  : .wav file → float32 audio array
      2. ASR   : audio → Rohingya Latin text  (facebook/mms-1b-all + rhg adapter)
      3. MT    : Rohingya text → English      (OpenRouter)
    """
    sample_rate, data = scipy.io.wavfile.read(wav_path)
    # Normalise to float32 [-1, 1] if the file contains integers
    if data.dtype != np.float32:
        data = data.astype(np.float32) / np.iinfo(data.dtype).max
    # Use first channel if stereo
    if data.ndim > 1:
        data = data[:, 0]

    rohingya_text = transcribe_rohingya(data, sample_rate)
    print(f"Transcribed Rohingya: {rohingya_text}")

    response = openrouter_client.chat.completions.create(
        model="google/gemini-3-pro-preview",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a translation assistant. "
                    "Translate the following Rohingya Latin script (Rohingyalish) text into English. "
                    "Return only the translated text with no explanations or extra formatting."
                ),
            },
            {"role": "user", "content": rohingya_text},
        ],
        max_tokens=1024
    )
    return response.choices[0].message.content.strip()


In [ ]:
import scipy
def translate_english_to_rhg_audio(english: str, file: str) -> np.ndarray:
    """
    Translate English text to Rohingya speech.

    Steps:
      1. MT  : English → Rohingya Latin text  (OpenRouter)
      2. TTS : Rohingya text → audio waveform (facebook/mms-tts-rhg)

    Returns:
      np.ndarray: float32 audio waveform at model.config.sampling_rate
    """
    rhg_text = translate_to_rohingya(english)
    print(f"Rohingya: {rhg_text}")

    inputs = tokenizer(rhg_text, return_tensors="pt")
    with torch.no_grad():
        waveform = model(**inputs).waveform

    audio_out = waveform.squeeze().detach().cpu().numpy().astype(np.float32)
    print("audio_out: ", audio_out)
    return scipy.io.wavfile.write(file, rate=model.config.sampling_rate, data=audio_out)

# --- Quick test ---
sample = "Welcome to the world of language technology."
audio_out = translate_english_to_rhg_audio(sample, "test.wav")


In [ ]:
def test_transcription(english_text, file_name):
    print("English text:", english_text)
    translate_english_to_rhg_audio(english_text, file_name)
    translation = translate_audio_to_english(file_name)
    print("Re-translated result:", translation)

test_transcription("All human beings are born free and equal in dignity and rights.", "test.wav")
test_transcription("Welcome to the world of language technology.", "test2.wav")
test_transcription("The quick brown fox jumps over the lazy dog.", "test3.wav")
test_transcription("You can buy rice and beans at the local grocery store.", "test4.wav")


In [ ]:
test_transcription("apple", "test5.wav")
test_transcription("can you tell me the meaning of this word?", "test6.wav")
test_transcription("Where is the nearest hospital?", "test7.wav")
test_transcription("give me a curry recipe", "test8.wav")